# Liane's Library – CRUD-Grundlagen

Kurzes Übungsnotebook für die ersten eigenen Abfragen gegen die Datenbank.

Statt die Verbindung hier nochmal komplett neu aufzubauen, wird `python/db.py`
importiert – dieselben Hilfsfunktionen (`query_dataframe`, `execute_statement`),
die auch die Streamlit-App verwendet. So gibt es nur eine Stelle im Projekt,
die weiß, wie die Verbindung aufgebaut wird.

Voraussetzung: Die Datenbank `lianes_library` ist bereits eingerichtet
(siehe README) und als Kernel ist die Conda-Umgebung `lianes-lib-env`
ausgewählt.

In [1]:
import db

db.test_connection()
print("Verbindung ok")

Verbindung ok


## 1. Lesen (READ)

`query_dataframe` gibt jede Abfrage direkt als pandas-DataFrame zurück –
praktisch zum Anschauen und Weiterverarbeiten.

In [2]:
df_books = db.query_dataframe("SELECT * FROM books ORDER BY title")
df_books

/Users/steve/lianes-diary/lianes-library/python/db.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, connection, params=params)


,book_id,title,author,isbn,category,shelf_location,notes,is_active,created_at,updated_at
0,4,Der Schwarm,Frank Schätzing,978-3-462-03462-3,Thriller,"Regal 2, Fach C",Echt tolles Buch! :),1,2026-07-23 09:37:28,2026-07-23 09:39:40
1,2,Der Vorleser,Bernhard Schlink,978-3-257-22953-1,Roman,"Regal 1, Fach A",NaN,1,2026-07-23 09:37:28,2026-07-23 09:37:28
2,1,Die Vermessung der Welt,Daniel Kehlmann,978-3-498-03528-3,Roman,"Regal 1, Fach A",Von Tante Erika geschenkt bekommen,1,2026-07-23 09:37:28,2026-07-23 09:37:28
3,3,Sofies Welt,Jostein Gaarder,978-3-423-12455-6,Sachbuch,"Regal 2, Fach B",Philosophiegeschichte für Einsteiger,1,2026-07-23 09:37:28,2026-07-23 09:37:28
4,5,Tschick,Wolfgang Herrndorf,978-3-499-25784-3,Jugendbuch,"Regal 3, Fach A",Klassenlektüre der 9. Klasse,1,2026-07-23 09:37:28,2026-07-23 09:37:28


In [3]:
df_borrowers = db.query_dataframe("SELECT * FROM borrowers ORDER BY name")
df_borrowers

/Users/steve/lianes-diary/lianes-library/python/db.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, connection, params=params)


,borrower_id,name,email,phone,relationship,notes,is_active,created_at,updated_at
0,1,Anna Bergmann,anna.bergmann@example.com,0170 1234567,Freundin,Liest gern Bücher über Berge,1,2026-07-23 09:37:28,2026-07-23 10:00:09
1,5,Julia Lehmann,julia.lehmann@example.com,NaN,Freundin,Aus dem Lesekreis,1,2026-07-23 09:37:28,2026-07-23 09:37:28
2,2,Markus Weber,markus.weber@example.com,0151 2345678,Kollege,Arbeitet im selben Büro,1,2026-07-23 09:37:28,2026-07-23 09:37:28
3,3,Sabine Hoffmann,sabine.hoffmann@example.com,0160 3456789,Familie,Cousine,1,2026-07-23 09:37:28,2026-07-23 09:37:28
4,4,Thomas Krüger,NaN,0176 4567890,Nachbar,NaN,1,2026-07-23 09:37:28,2026-07-23 09:37:28


In [4]:
# Die View v_open_loans übernimmt bereits die Joins zwischen loans, books und borrowers
df_open_loans = db.query_dataframe("SELECT * FROM v_open_loans ORDER BY due_date")
df_open_loans

/Users/steve/lianes-diary/lianes-library/python/db.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, connection, params=params)


,loan_id,book_id,title,author,borrower_id,borrower_name,borrower_email,borrower_phone,loan_date,due_date,loan_status
0,2,2,Der Vorleser,Bernhard Schlink,2,Markus Weber,markus.weber@example.com,0151 2345678,2026-06-15,2026-07-06,Overdue
1,3,4,Der Schwarm,Frank Schätzing,3,Sabine Hoffmann,sabine.hoffmann@example.com,0160 3456789,2026-07-01,2026-07-15,Overdue
2,5,3,Sofies Welt,Jostein Gaarder,5,Julia Lehmann,julia.lehmann@example.com,NaN,2026-07-10,2026-07-24,Loaned


## 2. Erstellen (CREATE)

Neues Buch anlegen. Die ISBN muss eindeutig sein (`uq_books_isbn` in
`sql/import.sql`) – ein zweiter Versuch mit derselben ISBN würde einen
Fehler werfen.

In [5]:
db.execute_statement(
    """
    INSERT INTO books (title, author, isbn, category, shelf_location)
    VALUES (%s, %s, %s, %s, %s)
    """,
    params=("Der Report der Magd", "Margaret Atwood", "9783596296328", "Dystopie", "Regal 4, Fach A"),
)
print("Buch angelegt")

Buch angelegt


In [6]:
neues_buch = db.query_dataframe(
    "SELECT * FROM books WHERE isbn = %s",
    params=("9783596296328",),
)
neues_buch

/Users/steve/lianes-diary/lianes-library/python/db.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, connection, params=params)


,book_id,title,author,isbn,category,shelf_location,notes,is_active,created_at,updated_at
0,9,Der Report der Magd,Margaret Atwood,9783596296328,Dystopie,"Regal 4, Fach A",None,1,2026-07-23 16:22:35,2026-07-23 16:22:35


## 3. Aktualisieren (UPDATE)

Die `book_id` aus der letzten Abfrage merken wir uns, damit wir gezielt
genau diesen einen Datensatz ändern.

In [7]:
book_id = int(neues_buch.iloc[0]["book_id"])

db.execute_statement(
    "UPDATE books SET shelf_location = %s WHERE book_id = %s",
    params=("Regal 4, Fach B", book_id),
)
print(f"book_id {book_id} aktualisiert")

book_id 9 aktualisiert


In [8]:
db.query_dataframe(
    "SELECT book_id, title, shelf_location FROM books WHERE book_id = %s",
    params=(book_id,),
)

/Users/steve/lianes-diary/lianes-library/python/db.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, connection, params=params)


,book_id,title,shelf_location
0,9,Der Report der Magd,"Regal 4, Fach B"


## 4. Löschen (DELETE)

In diesem Projekt wird nie hart gelöscht. Stattdessen setzen wir
`is_active = FALSE` (Soft Delete) – die App zeigt das Buch dann nicht mehr
an, die Ausleihhistorie bleibt aber erhalten.

In [9]:
db.execute_statement(
    "UPDATE books SET is_active = FALSE WHERE book_id = %s",
    params=(book_id,),
)

db.query_dataframe(
    "SELECT book_id, title, is_active FROM books WHERE book_id = %s",
    params=(book_id,),
)

/Users/steve/lianes-diary/lianes-library/python/db.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, connection, params=params)


,book_id,title,is_active
0,9,Der Report der Magd,0


## 5. Aufräumen

Das Testbuch aus diesem Notebook wollen wir nicht dauerhaft in der
Datenbank behalten. Deshalb hier ausnahmsweise ein echtes `DELETE` –
die App selbst bietet das absichtlich nicht an.

In [10]:
db.execute_statement("DELETE FROM books WHERE book_id = %s", params=(book_id,))
print("Testbuch entfernt")

Testbuch entfernt


## Nächste Schritte

Ein paar eigene Abfragen zum Weiterüben:

- Wie viele Bücher gibt es je Kategorie?
- Welche Person hat aktuell die meisten offenen Ausleihen?
- Welche Ausleihen sind überfällig (`v_loan_overview`, `loan_status = 'Overdue'`)?